
WHAT WE ARE TRYING TO FIND IN PHASE 3

We are answering:

“What factors increase the probability of an insurance claim?”

Not cost yet — just risk of claim.



In [ ]:
import pandas as pd

claims = pd.read_csv("/content/Insurance claims data.csv")


In [ ]:
claims.head()
claims.shape
claims.columns

Index(['policy_id', 'subscription_length', 'vehicle_age', 'customer_age',
       'region_code', 'region_density', 'segment', 'model', 'fuel_type',
       'max_torque', 'max_power', 'engine_type', 'airbags', 'is_esc',
       'is_adjustable_steering', 'is_tpms', 'is_parking_sensors',
       'is_parking_camera', 'rear_brakes_type', 'displacement', 'cylinder',
       'transmission_type', 'steering_type', 'turning_radius', 'length',
       'width', 'gross_weight', 'is_front_fog_lights', 'is_rear_window_wiper',
       'is_rear_window_washer', 'is_rear_window_defogger', 'is_brake_assist',
       'is_power_door_locks', 'is_central_locking', 'is_power_steering',
       'is_driver_seat_height_adjustable', 'is_day_night_rear_view_mirror',
       'is_ecw', 'is_speed_alert', 'ncap_rating', 'claim_status'],
      dtype='object')

CHECK CLAIM RATE

What we are doing:
See how many policies actually have claims.

Why:
This tells us:

how risky the portfolio is


In [ ]:
claims["claim_status"].value_counts(normalize=True)


,proportion
claim_status,
0,0.936032
1,0.063968


CREATE CLAIM FLAG (TARGET VARIABLE)

What we are doing:
Create a clean binary target variable.

Why:
We need a clear Yes / No claim indicator for:

- risk analysis
- later modelling
- dashboards

In [ ]:
claims["CLAIM_FLAG"] = claims["claim_status"].map(
    {1: "Claim", 0: "No_Claim"}
)


In [ ]:
claims[["claim_status", "CLAIM_FLAG"]].head()


,claim_status,CLAIM_FLAG
0,0,No_Claim
1,0,No_Claim
2,0,No_Claim
3,0,No_Claim
4,0,No_Claim


VEHICLE AGE BAND (RISK DRIVER)

What we are doing:
Group vehicles by age.

Why:
Older vehicles generally:

- break more
- have higher claim probability
- increase insurer risk
- This is one of the strongest auto-insurance risk drivers.

In [ ]:
claims["VEHICLE_AGE_BAND"] = pd.cut(
    claims["vehicle_age"],
    bins=[0, 1, 3, 5, 10, 50],
    labels=["New", "Almost_New", "Mid_Age", "Old", "Very_Old"]
)


In [ ]:
claims[["vehicle_age", "VEHICLE_AGE_BAND"]].head()


,vehicle_age,VEHICLE_AGE_BAND
0,1.2,Almost_New
1,1.8,Almost_New
2,0.2,New
3,0.4,New
4,1.0,New


CUSTOMER AGE BAND (DRIVER RISK)

What we are doing:
Group drivers by age.

Why:
- Driver age is a key claim-risk driver:
- Young drivers → higher accident risk
-Middle-aged → lower risk
- Older drivers → risk increases again

In [ ]:
claims["CUSTOMER_AGE_BAND"] = pd.cut(
    claims["customer_age"],
    bins=[0, 25, 40, 60, 100],
    labels=["Young", "Mid", "Senior", "Elder"]
)


In [ ]:
claims[["customer_age", "CUSTOMER_AGE_BAND"]].head()


,customer_age,CUSTOMER_AGE_BAND
0,41,Senior
1,35,Mid
2,44,Senior
3,44,Senior
4,56,Senior


Convert safety columns to numbers (0 / 1)

What we are doing:
Force all safety features to numeric.

Why:
We need numbers to calculate a score.

What this does:

Converts text → numbers where possible

Any non-numeric becomes 0

Safe and standard in insurance analytics

In [ ]:
for col in safety_features:
    claims[col] = pd.to_numeric(claims[col], errors="coerce").fillna(0)


In [ ]:
claims["SAFETY_SCORE"] = claims[safety_features].sum(axis=1)


In [ ]:
claims[safety_features + ["SAFETY_SCORE"]].head()


,airbags,is_esc,is_tpms,is_parking_sensors,is_parking_camera,is_brake_assist,SAFETY_SCORE
0,6,0.0,0.0,0.0,0.0,0.0,6.0
1,2,0.0,0.0,0.0,0.0,0.0,2.0
2,6,0.0,0.0,0.0,0.0,0.0,6.0
3,2,0.0,0.0,0.0,0.0,0.0,2.0
4,2,0.0,0.0,0.0,0.0,0.0,2.0


REGION RISK BAND

What we are doing:
Group regions by how risky they are.

Why:
Accident rates differ by geography (traffic, density, driving behavior).

We’ll use region_density as a proxy for risk.

In [ ]:
claims["REGION_RISK_BAND"] = pd.qcut(
    claims["region_density"],
    3,
    labels=["Low_Risk", "Medium_Risk", "High_Risk"]
)


In [ ]:
claims[["region_density", "REGION_RISK_BAND"]].head()


,region_density,REGION_RISK_BAND
0,8794,Low_Risk
1,27003,Medium_Risk
2,8794,Low_Risk
3,73430,High_Risk
4,5410,Low_Risk


SAVE PHASE 3 OUTPUT

What we are doing:
Save the claims risk feature table.

Why:
This file will be used for:

risk modelling

pricing decisions

portfolio risk analysis

Power BI dashboards

In [ ]:
claims.to_csv(
    "/content/claims_risk_features_phase3.csv",
    index=False
)
